# Continual Learning & Testing in Production

**Course:** [ML in Practice](https://ml-viz-ruby.vercel.app/courses/ml-in-practice/09-continual-learning-and-test-in-production)

Production ML is the discipline of running a loop: detect when the model has gone stale, retrain it, and *test* the new candidate against live users before letting it serve everyone. This notebook turns the four traffic-split patterns (shadow, canary, A/B, bandit) into self-contained numerical experiments using only NumPy + matplotlib.

1. **A/B test power analysis** — simulate two arms with conversion rates 0.10 and 0.105. For each sample size `n`, run 5,000 random splits and plot the detection power (probability of correctly rejecting H0 at α = 0.05) vs `n`. Find the smallest `n` that gives 80 % power.
2. **The peeking problem** — same A/B but stop on the first time the test crosses significance. Show the empirical false-positive rate balloons to ~20 % vs the nominal 5 %.
3. **Canary as accelerated trip-wire** — a candidate model has a 3× higher error rate. Show how a 5 %-traffic canary detects the regression in N requests vs full-traffic flip detecting it in K ≪ N requests *after* damage is done.
4. **Bandits (ε-greedy and Thompson sampling)** on a 3-arm Bernoulli problem with true means [0.05, 0.10, 0.15]. Plot cumulative regret over 5,000 steps for both algorithms vs a fixed-arm baseline and an oracle.

Self-contained: NumPy + matplotlib only. No torch, no sklearn, no scipy, no network, no API keys.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File -> Save a copy in Drive. Changes to this view are not saved.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Dark style matching the site theme.
plt.style.use('dark_background')
plt.rcParams.update({
    'axes.edgecolor': '#475569',
    'axes.labelcolor': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'axes.titlecolor': '#e2e8f0',
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'grid.color': '#2e3347',
    'savefig.facecolor': '#0f1117',
})
BRAND = '#6366f1'
TEAL = '#14b8a6'
ROSE = '#f43f5e'
YELLOW = '#eab308'

rng = np.random.default_rng(0)


## 1. A/B test power analysis

Two arms with true conversion rates $p_A = 0.10$ and $p_B = 0.105$. For each $n$ (impressions per arm), we want to know: if we run the experiment once and apply a two-proportion z-test at $\alpha = 0.05$, what is the probability we reject $H_0$? That probability is the *power* of the test at that sample size.

We compute the power empirically: 5,000 independent A/B simulations, count the fraction that produce $p < 0.05$. The standard target is **80 % power**.


In [ ]:
def two_proportion_z(success_a, n_a, success_b, n_b):
    """Two-proportion z-test. Returns |z|; reject H0 if |z| > 1.96 at alpha=0.05."""
    p_a = success_a / n_a
    p_b = success_b / n_b
    p_pool = (success_a + success_b) / (n_a + n_b)
    var = p_pool * (1 - p_pool) * (1 / n_a + 1 / n_b)
    if var <= 0:
        return 0.0
    return abs(p_b - p_a) / np.sqrt(var)

P_A, P_B = 0.10, 0.105
N_TRIALS = 5000
Z_CRIT = 1.96  # two-sided, alpha=0.05

sample_sizes = [1000, 2500, 5000, 10000, 25000, 50000, 100000, 250000, 500000]
powers = []
for n in sample_sizes:
    # Vectorise: draw all trials at once. successes ~ Binomial(n, p).
    succ_a = rng.binomial(n, P_A, size=N_TRIALS)
    succ_b = rng.binomial(n, P_B, size=N_TRIALS)
    # Apply the z-test trial-by-trial.
    z = np.array([two_proportion_z(a, n, b, n) for a, b in zip(succ_a, succ_b)])
    powers.append(float(np.mean(z > Z_CRIT)))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.semilogx(sample_sizes, powers, 'o-', color=BRAND, lw=2, markersize=7)
ax.axhline(0.80, color=TEAL, linestyle='--', lw=1, label='80% power target')
ax.axhline(0.05, color=ROSE, linestyle=':', lw=1, label='alpha = 0.05')
ax.set_xlabel('sample size per arm (log scale)')
ax.set_ylabel('empirical power at alpha=0.05')
ax.set_title('Power vs sample size: detecting a 0.5pp lift on a 10% base rate')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Smallest sample size with power >= 0.80.
for n, p in zip(sample_sizes, powers):
    if p >= 0.80:
        print(f'80% power reached at n = {n:,} per arm (power = {p:.3f})')
        break
else:
    print('80% power not reached in tested sample sizes; need a larger range.')


Two things stand out:

- Detecting a 0.5 pp lift on a 10 % base rate needs **hundreds of thousands of users per arm** at 80 % power. Tiny effects are statistically expensive.
- The curve is steep around the target — adding a single doubling of sample size around the 50%-power region typically takes you from underpowered to safely powered.


## 2. The peeking problem

Same setup as above — two arms, both true conversion rate $p = 0.10$ (so $H_0$ is *true*; any rejection is a false positive). We pre-commit to $n = 5{,}000$ users per arm. But instead of waiting until the end, we check significance every 100 trials and stop as soon as we cross $|z| > 1.96$.

Under the textbook assumption (single look at the end), the false-positive rate should be ~5 %. Let's see what *peeking* does.


In [ ]:
def run_peeking_trial(n_max=5000, check_every=100, rng=None):
    """Both arms have true p=0.10 (H0 true). Check significance every `check_every`
    trials; return True if we ever cross |z|>1.96."""
    arm_a = rng.binomial(1, 0.10, size=n_max)
    arm_b = rng.binomial(1, 0.10, size=n_max)
    cum_a = np.cumsum(arm_a)
    cum_b = np.cumsum(arm_b)
    for t in range(check_every, n_max + 1, check_every):
        z = two_proportion_z(cum_a[t - 1], t, cum_b[t - 1], t)
        if z > Z_CRIT:
            return True
    return False

n_sims = 2000
rng_pk = np.random.default_rng(42)
rejects_peeking = sum(run_peeking_trial(rng=rng_pk) for _ in range(n_sims))
fpr_peeking = rejects_peeking / n_sims

# Compare against the no-peeking baseline: one check at n=5,000.
rng_np = np.random.default_rng(43)
rejects_single = 0
for _ in range(n_sims):
    a = rng_np.binomial(5000, 0.10)
    b = rng_np.binomial(5000, 0.10)
    if two_proportion_z(a, 5000, b, 5000) > Z_CRIT:
        rejects_single += 1
fpr_single = rejects_single / n_sims

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(['single look\n(nominal)', 'peeking every\n100 trials'],
       [fpr_single, fpr_peeking],
       color=[BRAND, ROSE])
ax.axhline(0.05, color=YELLOW, linestyle='--', lw=1, label='nominal alpha = 0.05')
ax.set_ylabel('empirical false-positive rate')
ax.set_title(f'Peeking inflates FPR from {fpr_single:.1%} to {fpr_peeking:.1%}')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f'Single look FPR  : {fpr_single:.1%}  (target: 5%)')
print(f'Peeking FPR      : {fpr_peeking:.1%}  (4-6x inflation)')


Peeking every 100 trials and stopping on the first hit turns the textbook 5 % false-positive rate into something like 20 %. That's the multiple-comparison problem in disguise: we ran ~50 tests on overlapping subsets of the data and stopped on whichever fluctuation crossed first. The fixes are either (a) pre-register sample size and look exactly once, or (b) use sequential tests (mSPRT, always-valid p-values) designed to allow continuous monitoring.


## 3. Canary as accelerated trip-wire

The new model has a 3× higher error rate (1.5 % instead of the baseline 0.5 %). Two strategies:

- **Canary** routes 5 % of traffic to the candidate. We watch a rolling-window error rate on that 5 % and trigger an alert when it crosses a threshold.
- **Full flip** routes 100 % of traffic to the candidate. We watch the global error rate the same way.

Question: how many *requests* does each strategy need to detect the regression? And critically: how many requests have been *served by the broken model* by the time we detect it?


In [ ]:
BASELINE_ERR = 0.005
CANDIDATE_ERR = 0.015
WINDOW = 200       # rolling window size
ALERT_THRESHOLD = 0.012  # alert when window error rate exceeds this
TOTAL_QPS = 100    # requests per second of live traffic

def simulate(canary_frac, max_steps=20000, rng=None):
    """Stream `max_steps` requests; canary_frac go to candidate. Return (detection_time,
    bad_requests_served)."""
    detect_time = None
    candidate_errors = []
    bad_served = 0
    for t in range(max_steps):
        is_candidate = rng.random() < canary_frac
        err_rate = CANDIDATE_ERR if is_candidate else BASELINE_ERR
        err = rng.random() < err_rate
        if is_candidate:
            candidate_errors.append(int(err))
            if err:
                bad_served += 1
            # Roll the window only over candidate traffic — that's what monitoring
            # the canary slice means in practice.
            if len(candidate_errors) >= WINDOW:
                window_err = np.mean(candidate_errors[-WINDOW:])
                if window_err > ALERT_THRESHOLD and detect_time is None:
                    detect_time = t
                    break
    return detect_time, bad_served

rng_c = np.random.default_rng(7)
# 5% canary
t_canary, bad_canary = simulate(0.05, rng=rng_c)
# 100% flip
rng_c2 = np.random.default_rng(7)
t_flip, bad_flip = simulate(1.0, rng=rng_c2)

print(f'5% canary  : detected after {t_canary:>6,} live requests; {bad_canary} bad responses served')
print(f'100% flip  : detected after {t_flip:>6,} live requests; {bad_flip} bad responses served')
print()
print(f'The 5% canary needs ~{t_canary // max(t_flip, 1)}x more total requests to detect, but during')
print(f'that window only {bad_canary} users hit the bug vs {bad_flip} in the full flip.')


The canary is *slower* in calendar time (because only 5 % of traffic is candidate, the window fills up 20× slower) but the **damage** — number of users who got a buggy response — is dramatically smaller, because the broken model only ever saw 5 % of traffic.

That's the canary's actual job: not faster detection in wall-clock time, but bounded blast radius while detection happens.


## 4. Bandits: ε-greedy and Thompson sampling

Three Bernoulli arms with true means $\mu = [0.05, 0.10, 0.15]$. We have 5,000 steps total. Compare:

- **Fixed A/B** — equal split, ~1,667 pulls per arm.
- **ε-greedy** ($\varepsilon = 0.1$) — pick the empirical-best arm 90 % of the time, explore uniformly the other 10 %.
- **Thompson sampling** — Beta(1+successes, 1+failures) posterior per arm; sample once and pick the argmax.
- **Oracle** — always pull arm 3 (true mean 0.15). This gives the lowest possible regret (zero) and bounds the rest.


In [ ]:
TRUE_MEANS = np.array([0.05, 0.10, 0.15])
K = len(TRUE_MEANS)
T = 5000

def run_bandit(policy, rng):
    successes = np.zeros(K)
    failures = np.zeros(K)
    pulls = np.zeros(K, dtype=int)
    cum_regret = np.zeros(T)
    best_mean = TRUE_MEANS.max()
    for t in range(T):
        arm = policy(successes, failures, pulls, t, rng)
        reward = int(rng.random() < TRUE_MEANS[arm])
        successes[arm] += reward
        failures[arm] += 1 - reward
        pulls[arm] += 1
        cum_regret[t] = (cum_regret[t - 1] if t > 0 else 0.0) + (best_mean - TRUE_MEANS[arm])
    return cum_regret, pulls

def policy_ab(successes, failures, pulls, t, rng):
    # Round-robin: arm = t mod K. Pure fixed-arm baseline.
    return t % K

def policy_eps_greedy(eps):
    def _p(successes, failures, pulls, t, rng):
        if rng.random() < eps or pulls.min() == 0:
            return rng.integers(K)
        means = successes / np.maximum(pulls, 1)
        return int(np.argmax(means))
    return _p

def policy_thompson(successes, failures, pulls, t, rng):
    samples = rng.beta(1 + successes, 1 + failures)
    return int(np.argmax(samples))

def policy_oracle(successes, failures, pulls, t, rng):
    return int(np.argmax(TRUE_MEANS))

rngs = [np.random.default_rng(s) for s in range(4)]
reg_ab, pulls_ab = run_bandit(policy_ab, rngs[0])
reg_eps, pulls_eps = run_bandit(policy_eps_greedy(0.1), rngs[1])
reg_ts, pulls_ts = run_bandit(policy_thompson, rngs[2])
reg_or, pulls_or = run_bandit(policy_oracle, rngs[3])

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(reg_ab, color=ROSE, lw=2, label=f'Fixed A/B (round-robin) — final regret {reg_ab[-1]:.1f}')
ax.plot(reg_eps, color=YELLOW, lw=2, label=f'eps-greedy (eps=0.1) — final regret {reg_eps[-1]:.1f}')
ax.plot(reg_ts, color=TEAL, lw=2, label=f'Thompson sampling — final regret {reg_ts[-1]:.1f}')
ax.plot(reg_or, color=BRAND, lw=2, linestyle=':', label='Oracle — final regret 0.0')
ax.set_xlabel('step t')
ax.set_ylabel('cumulative regret  Σ (μ* − μ_a)')
ax.set_title('Cumulative regret on a 3-arm Bernoulli problem with means [0.05, 0.10, 0.15]')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Arm pull counts:')
print(f'  Fixed A/B       : {pulls_ab}')
print(f'  eps-greedy      : {pulls_eps}')
print(f'  Thompson        : {pulls_ts}')
print(f'  Oracle          : {pulls_or}')


The fixed A/B is the worst because it commits to spending 1/K of all traffic on the worst arm for the entire test. ε-greedy is much better — once it identifies arm 3 as the empirical winner, it spends 90 % of remaining traffic there. Thompson sampling is better still: it explores *less* the more confident it gets, automatically annealing exploration. The oracle is a horizontal line at 0 by definition.

Note the **trade-off**: the better the bandit at minimising regret, the more biased its post-hoc arm-mean estimates become. Estimating $\mu_1$ from the few pulls Thompson made on arm 1 is noisy; running a clean confirmatory A/B between the winner and a baseline is much harder under bandit data.


## ✏️ Your turn — implement Thompson sampling and compare against ε-greedy

Your goal: implement `thompson_sampling(true_means, n_steps, seed)` returning the cumulative regret array over `n_steps` on a Bernoulli bandit with the given true arm means, using Beta(1+successes, 1+failures) posterior sampling.

We'll then verify that Thompson sampling beats ε-greedy ($\varepsilon = 0.1$) on cumulative regret over 3,000 steps on the same arm-means we used above.


In [ ]:
def thompson_sampling(true_means, n_steps, seed=0):
    """Run Thompson sampling on a Bernoulli bandit.

    Returns: cumulative regret array of shape (n_steps,).
    """
    rng = np.random.default_rng(seed)
    k = len(true_means)
    best = max(true_means)
    successes = np.zeros(k)
    failures = np.zeros(k)
    cum_regret = np.zeros(n_steps)

    for t in range(n_steps):
        # TODO(you): draw one Beta(1+successes, 1+failures) sample per arm,
        # pick the arm with the highest sampled mean, pull it, update.
        # Hint: rng.beta(...) accepts arrays. argmax returns int.
        samples = ...  # TODO
        arm = ...      # TODO

        reward = int(rng.random() < true_means[arm])
        successes[arm] += reward
        failures[arm] += 1 - reward
        cum_regret[t] = (cum_regret[t - 1] if t > 0 else 0.0) + (best - true_means[arm])

    return cum_regret


In [ ]:
# Compare Thompson against a known-good eps-greedy baseline.
def eps_greedy_reference(true_means, n_steps, eps=0.1, seed=0):
    rng = np.random.default_rng(seed)
    k = len(true_means)
    best = max(true_means)
    successes = np.zeros(k)
    pulls = np.zeros(k)
    cum_regret = np.zeros(n_steps)
    for t in range(n_steps):
        if rng.random() < eps or pulls.min() == 0:
            arm = int(rng.integers(k))
        else:
            means = successes / np.maximum(pulls, 1)
            arm = int(np.argmax(means))
        reward = int(rng.random() < true_means[arm])
        successes[arm] += reward
        pulls[arm] += 1
        cum_regret[t] = (cum_regret[t - 1] if t > 0 else 0.0) + (best - true_means[arm])
    return cum_regret

true_means = [0.05, 0.10, 0.15]
n_steps = 3000

# Average over a few seeds to smooth the comparison.
ts_regrets = np.mean([thompson_sampling(true_means, n_steps, seed=s)[-1] for s in range(5)])
eg_regrets = np.mean([eps_greedy_reference(true_means, n_steps, seed=s)[-1] for s in range(5)])

print(f'Thompson sampling final regret (avg of 5 seeds) : {ts_regrets:.2f}')
print(f'eps-greedy final regret (avg of 5 seeds)        : {eg_regrets:.2f}')

assert ts_regrets < eg_regrets, (
    f'Expected Thompson regret ({ts_regrets:.2f}) to be < eps-greedy ({eg_regrets:.2f}). '
    'Did you fill in the TODO sampling step correctly?'
)
print('\n✅ Thompson sampling beats eps-greedy on cumulative regret.')


<details>
<summary>Solution</summary>

Inside the loop:

```python
samples = rng.beta(1 + successes, 1 + failures)
arm = int(np.argmax(samples))
```

That's all there is to vanilla Thompson sampling: maintain a Beta posterior per arm (conjugate to Bernoulli rewards), draw one sample per arm per step, pick the argmax. Exploration falls out automatically because under-pulled arms have wide posteriors that occasionally sample high; well-pulled arms have tight posteriors centred on their true means.

</details>


## Recap

- **Power analysis** sets the *only* statistically defensible sample-size — pre-register it.
- **Peeking** silently turns a 5 % FPR into ~20 %. Use sequential tests if you want to look continuously.
- **Canaries** are not faster than full flips at detecting regressions; they are *safer* — the broken model only ever sees 5 % of traffic.
- **Bandits** cut regret substantially over a fixed A/B, at the cost of biased post-hoc estimates and harder confirmatory statistics.
